# Phase 2 — Theoretical Foundation

**Course:** AML-DL Project | **Authors:** Prerak Arya (230039), Saikiran Bompelliwar (230046)

This notebook presents the mathematical foundations underlying our Phase 2 models. We derive key formulations from first principles, connecting the mathematical theory to our implementation choices.

---

## 1. One-Class SVM: Primal and Dual Formulation

### 1.1 Problem Setup

Given training data $\{x_1, \ldots, x_n\} \subset \mathbb{R}^d$ consisting exclusively of normal traffic samples, we seek a function $f: \mathbb{R}^d \to \{+1, -1\}$ that outputs $+1$ for "normal" data and $-1$ for "anomalous" data. The core challenge: we have no anomalous examples during training.

### 1.2 Primal Problem (Schölkopf et al., 2001)

The idea: map data into a feature space $\mathcal{H}$ via $\Phi: \mathbb{R}^d \to \mathcal{H}$ and find a hyperplane that separates the mapped data from the origin with maximum margin:

$$\min_{w \in \mathcal{H}, \rho \in \mathbb{R}, \xi \in \mathbb{R}^n} \frac{1}{2}\|w\|^2 + \frac{1}{\nu n}\sum_{i=1}^n \xi_i - \rho$$

$$\text{subject to: } \quad w \cdot \Phi(x_i) \geq \rho - \xi_i, \quad \xi_i \geq 0 \quad \forall i$$

**Interpretation of each term:**
- $\frac{1}{2}\|w\|^2$: Margin maximisation (standard SVM regularisation)
- $\frac{1}{\nu n}\sum_i \xi_i$: Penalty for slack (data points on wrong side of hyperplane)
- $-\rho$: We want $\rho$ to be large (hyperplane far from origin), so we subtract it to minimise

**The role of $\nu$:** It controls the trade-off between the fraction of training data allowed to be on the wrong side of the hyperplane (potential outliers in normal data) and the margin width. Specifically:
- $\nu$ is an upper bound on the fraction of outliers (samples with $\xi_i > 0$)
- $\nu$ is a lower bound on the fraction of support vectors
- Setting $\nu = 0.05$ means at most 5% of normal training data may be treated as anomalous

### 1.3 Dual Derivation via Lagrange Multipliers

Introducing Lagrange multipliers $\alpha_i \geq 0$ for the inequality constraints and $\beta_i \geq 0$ for $\xi_i \geq 0$:

$$\mathcal{L} = \frac{1}{2}\|w\|^2 + \frac{1}{\nu n}\sum_i \xi_i - \rho - \sum_i \alpha_i(w \cdot \Phi(x_i) - \rho + \xi_i) - \sum_i \beta_i \xi_i$$

**KKT conditions** (setting partial derivatives to zero):

$$\frac{\partial \mathcal{L}}{\partial w} = 0 \implies w = \sum_i \alpha_i \Phi(x_i)$$

$$\frac{\partial \mathcal{L}}{\partial \rho} = 0 \implies \sum_i \alpha_i = 1$$

$$\frac{\partial \mathcal{L}}{\partial \xi_i} = 0 \implies \alpha_i + \beta_i = \frac{1}{\nu n} \implies 0 \leq \alpha_i \leq \frac{1}{\nu n}$$

Substituting back into the Lagrangian, we obtain the **dual problem**:

$$\min_{\alpha} \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j K(x_i, x_j) \quad \text{s.t.} \quad 0 \leq \alpha_i \leq \frac{1}{\nu n}, \quad \sum_i \alpha_i = 1$$

where $K(x_i, x_j) = \Phi(x_i) \cdot \Phi(x_j)$ is the kernel function.

**Decision function:** $f(x) = \text{sgn}\left(\sum_{i \in SV} \alpha_i K(x_i, x) - \rho\right)$

### 1.4 RBF Kernel Properties

The Radial Basis Function kernel:

$$K(x_i, x_j) = \exp\left(-\gamma \|x_i - x_j\|^2\right)$$

Properties relevant to our task:
- **Infinite-dimensional feature space:** The RBF kernel corresponds to a feature map into $\ell^2$ (infinite-dimensional), enabling arbitrarily complex decision boundaries
- **Local similarity:** $K(x_i, x_j) \to 1$ as $x_i \to x_j$ and $K \to 0$ as $\|x_i - x_j\| \to \infty$
- **Universal approximation:** With enough support vectors, any continuous decision boundary can be approximated
- **Scale heuristic:** `gamma='scale'` sets $\gamma = \frac{1}{d \cdot \text{Var}(X)}$, normalising for feature count and variance

---

## 2. Autoencoder with Skip Connections

### 2.1 Standard Autoencoder Objective

An autoencoder consists of an encoder $E: \mathbb{R}^d \to \mathbb{R}^k$ and decoder $D: \mathbb{R}^k \to \mathbb{R}^d$ with $k < d$ (bottleneck). The training objective:

$$\min_{E, D} \frac{1}{n}\sum_{i=1}^n \mathcal{L}(x_i, D(E(x_i)))$$

For MSE loss: $\mathcal{L}(x, \hat{x}) = \|x - \hat{x}\|^2 = \sum_{j=1}^d (x_j - \hat{x}_j)^2$

**Connection to PCA:** For linear autoencoders (no activation functions) with MSE loss, the optimal solution spans the same subspace as the top-$k$ principal components (Baldi & Hornik, 1989). Non-linear activations allow the autoencoder to learn **curved manifolds** rather than flat subspaces — essential for the non-linear structure of network traffic data.

### 2.2 Weighted Reconstruction Loss

We modify the standard MSE to weight features by their inverse variance:

$$\mathcal{L}_\text{weighted} = \frac{1}{n}\sum_{i=1}^n \sum_{j=1}^d w_j(x_{ij} - \hat{x}_{ij})^2$$

where $w_j = \frac{1}{\text{Var}(x_j) + \epsilon}$, normalised so $\frac{1}{d}\sum_j w_j = 1$.

**Mathematical justification:** Under the assumption that each feature follows an independent Gaussian distribution $x_j \sim \mathcal{N}(\mu_j, \sigma_j^2)$ for normal data, the Mahalanobis distance (optimal anomaly score for Gaussian data) weights each feature by $1/\sigma_j^2$. Our inverse-variance weighting approximates the Mahalanobis metric within the reconstruction loss, bridging statistical anomaly detection theory with deep learning practice.

**Gradient of weighted loss w.r.t. output layer:**

$$\frac{\partial \mathcal{L}_\text{weighted}}{\partial \hat{x}_{ij}} = -\frac{2w_j}{n}(x_{ij} - \hat{x}_{ij})$$

This means features with higher weight (low variance) produce stronger gradients when reconstructed incorrectly, forcing the network to prioritise their accurate reconstruction.

### 2.3 Skip Connection Mechanics

Our architecture adds skip connections between symmetric encoder-decoder layers. Let $h_l^\text{enc}$ denote encoder layer $l$'s output and $h_l^\text{dec}$ the corresponding decoder layer's output. With skip connections:

$$h_l^\text{dec} = g\left(W_l \left[\text{dec}_l(h_{l-1}^\text{dec}) \;\|\; h_{L-l}^\text{enc}\right] + b_l\right)$$

where $[\cdot \| \cdot]$ denotes concatenation and $g$ is the activation function.

**Gradient analysis:** The gradient from the loss to encoder layer $l$ decomposes into two paths:

$$\frac{\partial \mathcal{L}}{\partial h_l^\text{enc}} = \underbrace{\frac{\partial \mathcal{L}}{\partial h_l^\text{dec}} \cdot \frac{\partial h_l^\text{dec}}{\partial h_l^\text{enc}}}_{\text{skip path (direct)}} + \underbrace{\frac{\partial \mathcal{L}}{\partial z} \cdot \frac{\partial z}{\partial h_l^\text{enc}}}_{\text{bottleneck path (deep)}}$$

The skip path provides a **short gradient path** (fewer layers to traverse), ensuring that encoder layers receive strong gradient signal even in deep architectures. The bottleneck path provides the **abstract representation learning** signal. Together, they enable the encoder to learn features useful for both fine-grained reconstruction and abstract compression.

### 2.4 Layer Details

| Layer | Dimensions | Purpose |
|-------|-----------|---------|
| **Encoder 1** | $43 \to 128$ | Expand to capture feature interactions |
| **BatchNorm** | 128 | Stabilise activations, reduce covariate shift |
| **LeakyReLU(0.2)** | 128 | Non-linearity; leak prevents dying neurons |
| **Dropout(0.2)** | 128 | Regularisation for small normal-only dataset |
| **Encoder 2** | $128 \to 64$ | Compress to intermediate representation |
| **Encoder 3** | $64 \to 32$ | Bottleneck — abstract normal patterns |
| **Decoder 1** | $32 \to 64$ | Begin reconstruction |
| **Skip + Decoder 2** | $64+64=128 \to 128$ | Concatenate enc2 output, refine |
| **Skip + Decoder 3** | $128+128=256 \to 43$ | Concatenate enc1 output, reconstruct |

**Why LeakyReLU over ReLU:** ReLU sets all negative pre-activations to zero, which can cause "dying neurons" — units that permanently output zero. LeakyReLU maintains a small slope ($\alpha = 0.2$) for negative inputs: $f(x) = \max(\alpha x, x)$. This is important for our task because negative feature values (after standardisation) are common and carry information.

**Why BatchNorm:** Network traffic features span vastly different scales (bytes: $10^0$ to $10^6$; rates: $[0, 1]$; counts: $10^0$ to $10^3$). Even after StandardScaler, internal activations can drift during training (internal covariate shift). BatchNorm normalises each layer's output to zero mean and unit variance, accelerating convergence.

---

## 3. Variational Autoencoder

### 3.1 Generative Model and ELBO Derivation

The VAE posits a generative model: a latent variable $z$ is drawn from a prior $p(z) = \mathcal{N}(0, I)$, and the observation $x$ is generated from $p(x|z)$ (parameterised by the decoder network). The marginal likelihood:

$$p(x) = \int p(x|z) p(z)\, dz$$

This integral is intractable for deep networks. We introduce a variational approximation $q_\phi(z|x) = \mathcal{N}(\mu_\phi(x), \sigma^2_\phi(x) I)$ (the encoder) and derive the ELBO:

**Step 1:** Write $\log p(x)$ using the variational posterior:

$$\log p(x) = \mathbb{E}_{q(z|x)}\left[\log \frac{p(x, z)}{q(z|x)}\right] + D_\text{KL}(q(z|x) \| p(z|x))$$

**Step 2:** Since $D_\text{KL} \geq 0$:

$$\log p(x) \geq \mathbb{E}_{q(z|x)}\left[\log \frac{p(x, z)}{q(z|x)}\right] = \text{ELBO}$$

**Step 3:** Expand the ELBO:

$$\text{ELBO} = \mathbb{E}_{q(z|x)}[\log p(x|z)] - D_\text{KL}(q(z|x) \| p(z))$$

The first term is the **expected reconstruction log-likelihood** (how well the decoder reconstructs $x$ from latent samples). The second term is the **KL divergence** between the approximate posterior and the prior (how close the latent distribution is to a standard Gaussian).

### 3.2 KL Divergence: Closed-Form Solution

For two Gaussians $q = \mathcal{N}(\mu, \text{diag}(\sigma^2))$ and $p = \mathcal{N}(0, I)$ with $J$-dimensional latent space:

$$D_\text{KL}(q \| p) = -\frac{1}{2}\sum_{j=1}^J \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

**Derivation sketch:** Using the general formula $D_\text{KL}(q \| p) = \int q(z) \log \frac{q(z)}{p(z)} dz$ and plugging in the Gaussian densities:

$$= \frac{1}{2}\left[\text{tr}(\Sigma) + \mu^T \mu - J - \log \det(\Sigma)\right]$$

For diagonal $\Sigma = \text{diag}(\sigma_1^2, \ldots, \sigma_J^2)$: $\text{tr}(\Sigma) = \sum_j \sigma_j^2$ and $\log \det(\Sigma) = \sum_j \log \sigma_j^2$, yielding our formula.

**Gradients of KL term:**

$$\frac{\partial D_\text{KL}}{\partial \mu_j} = \mu_j$$

$$\frac{\partial D_\text{KL}}{\partial \log \sigma_j^2} = \frac{1}{2}(\sigma_j^2 - 1)$$

These gradients push $\mu \to 0$ and $\sigma^2 \to 1$, i.e., the posterior toward the prior. The reconstruction loss opposes this — it needs informative latent codes. The $\beta$ parameter controls which force dominates.

### 3.3 Reparameterisation Trick

Sampling $z \sim q(z|x) = \mathcal{N}(\mu, \sigma^2 I)$ is not differentiable with respect to $\mu$ and $\sigma$. The reparameterisation trick rewrites:

$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

Now $z$ is a deterministic function of $\mu$, $\sigma$, and the noise $\epsilon$. Gradients flow through $\mu$ and $\sigma$ normally, while $\epsilon$ is treated as a constant input. This is the key technical contribution of Kingma & Welling (2014) that makes VAE training practical.

### 3.4 $\beta$-VAE and the Reconstruction-Regularity Trade-off

$$\mathcal{L}_{\beta\text{-VAE}} = \underbrace{\frac{1}{n}\sum_{i=1}^n \|x_i - \hat{x}_i\|^2}_{\text{reconstruction (MSE)}} + \beta \cdot \underbrace{D_\text{KL}(q(z|x) \| p(z))}_{\text{regularity}}$$

| $\beta$ | Behaviour | Use Case |
|---------|-----------|----------|
| $\beta \gg 1$ | Strong latent regularity, poor reconstruction | Disentangled representation learning |
| $\beta = 1$ | Standard VAE (ELBO maximisation) | General purpose |
| $\beta < 1$ | High reconstruction fidelity, weaker regularity | **Anomaly detection (our choice: $\beta = 0.5$)** |
| $\beta = 0$ | Deterministic autoencoder (no latent regularisation) | Degenerate case |

**Why $\beta = 0.5$ for anomaly detection:** The anomaly score is based on reconstruction error. Stronger reconstruction produces larger gaps between normal (low error) and anomalous (high error) samples. Setting $\beta < 1$ biases the model toward reconstruction quality. We choose $\beta = 0.5$ as a moderate setting — the latent space retains some structure (useful for Phase 3 hybrid models) while prioritising detection performance.

### 3.5 Reconstruction Probability (An & Cho, 2015)

Standard anomaly scoring: $\text{score}(x) = \|x - D(E(x))\|^2$ (single-point estimate)

Reconstruction probability scoring:

$$\text{score}(x) = \frac{1}{L}\sum_{l=1}^L \|x - D(z^{(l)})\|^2, \quad z^{(l)} \sim \mathcal{N}(\mu(x), \sigma^2(x) I)$$

**Statistical properties:** For normal data with well-determined latent codes (low $\sigma$), the $L$ samples cluster around $\mu$, producing consistent low scores. For anomalous data:
- If the encoder is uncertain ($\sigma$ is large): samples spread widely, reconstructions vary, average error is high
- If the encoder maps to an unusual region ($\mu$ is extreme): even consistent samples reconstruct poorly

This dual mechanism makes reconstruction probability more discriminative than point estimates.

---

## 4. Optimisation

### 4.1 Adam Optimiser

Adam (Kingma & Ba, 2015) maintains exponential moving averages of the gradient ($m_t$) and squared gradient ($v_t$):

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t \quad \text{(first moment / momentum)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2 \quad \text{(second moment / adaptive LR)}$$

Bias-corrected estimates:
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

Parameter update:
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon}\hat{m}_t$$

**Why Adam for this task:**
- **Adaptive learning rates:** Network traffic features have heterogeneous magnitudes → parameters connected to high-variance features need smaller updates (Adam provides this automatically through $v_t$)
- **Momentum:** Smooths gradient signal during curriculum stage transitions, preventing catastrophic forgetting of patterns learned in earlier stages
- **Bias correction:** At the start of each curriculum stage, $m_t$ and $v_t$ are stale from the previous data distribution → bias correction prevents distorted updates

### 4.2 Learning Rate Scheduling

We use `ReduceLROnPlateau` — reduce LR by factor 0.5 when validation loss plateaus for 5 epochs. This annealing strategy:
1. **Early training (high LR):** Rapidly explores the loss landscape, finding the basin of attraction
2. **Later training (reduced LR):** Fine-tunes within the basin, avoiding overshooting

### 4.3 Weight Decay (L2 Regularisation)

We add $\lambda \|W\|_F^2$ to the loss ($\lambda = 10^{-5}$), equivalent to a Gaussian prior on the weights: $W \sim \mathcal{N}(0, \frac{1}{\lambda}I)$. This prevents weights from growing large and overfitting to the normal-only training set.

### 4.4 Early Stopping

We monitor validation loss and stop training when it hasn't improved for $p$ epochs (patience). This implements **implicit regularisation** — the model is trained just long enough to learn the normal manifold without memorising training noise.

---

## 5. Information-Theoretic Perspective

### 5.1 Information Bottleneck

The autoencoder bottleneck implements a form of the Information Bottleneck (Tishby et al., 2000):

$$\min_{E, D} I(X; Z) \quad \text{subject to} \quad I(Z; \hat{X}) \geq I_0$$

where $I(\cdot; \cdot)$ denotes mutual information. The bottleneck dimension ($k = 32$ for AE, $k = 16$ for VAE) limits $I(X; Z)$ — the model must compress the input, retaining only the most essential structure. For anomaly detection, this compression forces the model to learn the **core normal manifold**, discarding noise and idiosyncratic variations. Anomalies that do not lie on this manifold are poorly reconstructed.

### 5.2 Rate-Distortion Theory Connection

The VAE loss can be interpreted through rate-distortion theory:
- **Rate** $R = D_\text{KL}(q \| p)$: bits of information the encoder transmits about $x$ through $z$
- **Distortion** $D = \mathbb{E}[\|x - \hat{x}\|^2]$: reconstruction quality

The $\beta$-VAE loss traces out the **rate-distortion curve**: $\min_q D + \beta R$. Lower $\beta$ allows higher rate (more information in $z$, better reconstruction). Our choice of $\beta = 0.5$ operates at a point on this curve that favours low distortion (high detection quality) at the cost of higher rate (less structured latent space).

---

*These mathematical foundations are implemented in the model training notebooks that follow. Each implementation decision maps directly to the theory presented here.*